

### Ziel dieser Datei
Benötigte Rohdaten für gesamte Schweiz runterladen. Dann filtern auf die relevanten Attribute (Wiese, Wälder, Schutzzonen usw). Dann auch sicherstellen, dass alle Datensätze in LV95 sind.

### Datenquellen
OSM-Daten über OverpassTurbo die jeweiligen Daten holen. Korrekte Abfragefilter machen, damit die richtigen Daten heruntergeladen werden!
  * Wiesen/Wälder (z.B. `landuse=meadow`, `landuse=forest`, usw.)
  * Infrastruktur (z.B. `amenity=farm`, `landuse=farmyard` für Bauernhöfe; `highway=bus_stop`, `railway=station` für ÖV)
  * Wenn möglich Geländeneigung, sonst mindestens natürliche Gefahrenzonen (`natural=cliff` für Felswände, `natural=scree` für Geröll) um dann diese Gebiete später auszuschliessen
Schutzgebiete Vektordatensatz als GeoJSON/Geopackage von geo.admin.ch herunterladen.

### Grober Codeaufbau
1. GeoJSON-Dateien einlesen
2. Koordinatensystem anpassen in LV95 (für berechnungen / verschnitte später)
3. Attributtabellen bereinigen: Unnötige Spalten löschen, damit die Dateien performant laufen.

### Export & Übernahme für die Nächste Datei 2
* Bereinigte Daten einzeln als GeoJSON abspeichern!
* Saubere Benennung nach Thematik, dass für Import in Datei 2 alles klar ist. z.B. `bearbeitet_flächen.geojson` / `bearbeitet_schutzgebiete.geojson`.

In [2]:
import osmnx as ox
import geopandas as gpd
import warnings
import pandas as pd

In [3]:
# Warnungen unterdrücken für eine saubere Ausgabe
warnings.filterwarnings('ignore')

In [4]:
# 1. OSM-Daten beziehen
place_name = "Kanton Basel-Landschaft, Switzerland"

print("Lade Wälder und Wiesen")
tags_nature = {'landuse': ['meadow', 'forest']}
gdf_nature = ox.features_from_place(place_name, tags_nature)

print("Lade Infrastruktur")
tags_infra = {
    'amenity': ['farm'],
    'landuse': ['farmyard'],
    'highway': ['bus_stop'],
    'railway': ['station'],
    'emergency': ['fire_hydrant']
}
gdf_infra = ox.features_from_place(place_name, tags_infra)

print("Lade Hydranten")
tags_hydrants = {'emergency': 'fire_hydrant'}
gdf_hydrants = ox.features_from_place(place_name, tags_hydrants)

print("Lade Gefahrenzonen")
tags_hazards = {'natural': ['cliff', 'scree']}
gdf_hazards = ox.features_from_place(place_name, tags_hazards)

print("\n Daten erfolgreich geladen")

Lade Wälder und Wiesen
Lade Infrastruktur
Lade Hydranten
Lade Gefahrenzonen

 Daten erfolgreich geladen


In [5]:
# 2. geo.admin aus src Daten laden
print("Lade Schutzgebiete und Wohnzohnen")
pfad_jagdbann = 'src/bundesinventare-jagdbanngebiete_2056.shp/N2023_Revision_jagdbann.shp'
pfad_moor = 'src/bundesinventare-moorlandschaften_2056.shp/N2017_Revision_Moorlandschaft_20171101.shp'
pfad_auen = 'src/auen-vegetationskarten_2056.gpkg'
pfad_wohngeb = 'src/wohngebiete-aulav_2056.gpkg'

gdf_jagdbann = gpd.read_file(pfad_jagdbann)
gdf_moor = gpd.read_file(pfad_moor)
gdf_auen = gpd.read_file(pfad_auen, layer='Auenvegetation')
gdf_wohngeb = gpd.read_file(pfad_wohngeb, layer='Bufferzone')

print("\n Daten erfolgreich geladen")

Lade Schutzgebiete und Wohnzohnen

 Daten erfolgreich geladen


In [6]:
# 3. Koordinatensystem in LV95 (EPSG:2056) kontrollieren/transformieren
gdf_nature_lv95 = gdf_nature.to_crs(epsg=2056)
gdf_infra_lv95 = gdf_infra.to_crs(epsg=2056)
gdf_hazards_lv95 = gdf_hazards.to_crs(epsg=2056)
gdf_hydrants_lv95 = gdf_hydrants.to_crs(epsg=2056)

gdf_jagdbann_lv95 = gdf_jagdbann.to_crs(epsg=2056)
gdf_moor_lv95 = gdf_moor.to_crs(epsg=2056)
gdf_auen_lv95 = gdf_auen.to_crs(epsg=2056)
gdf_wohngeb_lv95 = gdf_wohngeb.to_crs(epsg=2056)

print("Koordinatentransformation erfolgreich")

Koordinatentransformation erfolgreich


In [7]:
# 4. Attributtabellen bereinigen

def clean_attributes(gdf, keep_columns):
    existing_cols = [col for col in keep_columns if col in gdf.columns] + ['geometry']
    return gdf[existing_cols]

gdf_nature_clean = clean_attributes(gdf_nature_lv95, ['landuse'])
gdf_infra_clean = clean_attributes(gdf_infra_lv95, ['amenity', 'landuse', 'highway', 'railway', 'name'])
gdf_hazards_clean = clean_attributes(gdf_hazards_lv95, ['natural'])
gdf_hydrants_clean = clean_attributes(gdf_hydrants_lv95, ['emergency'])

# Hydrant in Infrastruktur ergänzen
gdf_infrastructure_clean = pd.concat([gdf_infra_clean, gdf_hydrants_clean], ignore_index=True)

print("Attributtabellen erfolgreich bereinigt")

Attributtabellen erfolgreich bereinigt


In [8]:
# Schutzgebiete zusammenfügen, attribute benennen
gdf_jagdbann_clean = clean_attributes(gdf_jagdbann_lv95, [])
gdf_jagdbann_clean['schutz_typ'] = 'Jagdbanngebiet'

gdf_moor_clean = clean_attributes(gdf_moor_lv95, [])
gdf_moor_clean['schutz_typ'] = 'Moorlandschaft'

gdf_auen_clean = clean_attributes(gdf_auen_lv95, [])
gdf_auen_clean['schutz_typ'] = 'Aue'

gdf_wohngeb_clean = clean_attributes(gdf_wohngeb_lv95, [])
gdf_wohngeb_clean['schutz_typ'] = 'Siedlung'

# Ein gemeinsamer Datensatz
gdf_schutz_kombiniert = pd.concat([gdf_jagdbann_clean, gdf_moor_clean, gdf_auen_clean, gdf_wohngeb_clean], ignore_index=True)
gdf_schutz_kombiniert = gpd.GeoDataFrame(gdf_schutz_kombiniert, geometry='geometry', crs="EPSG:2056")

In [ ]:
# 5. Export als bereinigte GeoJSON-Dateien für räumliche Analyse
gdf_nature_clean.to_file("data_BL/bearbeitet_flaechen.geojson", driver="GeoJSON")
gdf_infrastructure_clean.to_file("data_BL/bearbeitet_infrastruktur.geojson", driver="GeoJSON")
gdf_hazards_clean.to_file("data_BL/bearbeitet_gefahrenzonen.geojson", driver="GeoJSON")
gdf_schutz_kombiniert.to_file("data_BL/bearbeitet_schutzgebiete.geojson", driver="GeoJSON")

print("Datenbezug und Export erfolgreich abgeschlossen!")

Datenbezug und Export erfolgreich abgeschlossen!
